# ADL Results Explorer

Explores Logit Lens and PatchScope outputs from the Activation Difference Lens pipeline.

In [1]:
from pathlib import Path
import matplotlib.pyplot as plt

# --- Configuration (edit these) ---
RESULTS_DIR = Path(
    "../../../adl_results/workspace/model-organisms/diffing_results/olmo2_1B_sft/examples_narrow-sft-2/activation_difference_lens"
)
LAYERS = [7, 14, 15]
DATASET = "tulu-3-sft-olmo-2-mixture"
LOGIT_LENS_POSITION = -1  # Position for per-position logit lens view
PATCHSCOPE_POSITION = -1  # Position for per-position patchscope view
N_POSITIONS = 128  # Total positions (config: n)
LOGIT_LENS_MAX_ROWS = 20  # Set to an integer to truncate logit lens tables
PATCHSCOPE_GRADER = "openai_gpt-5-mini"
MODEL_ID = "allenai/OLMo-2-0425-1B-DPO"

LAYER_DIRS = {layer: RESULTS_DIR / f"layer_{layer}" / DATASET for layer in LAYERS}

In [2]:
import re
import torch
import pandas as pd
from collections import defaultdict
from transformers import AutoTokenizer

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 60)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)


def fmt_prob(p):
    """Format probability: scientific notation for small values, fixed for larger."""
    if abs(p) < 0.01:
        return f"{p:.2e}"
    return f"{p:.4f}"


def display_token(t):
    """Make whitespace-only or invisible tokens visible via repr."""
    if not t.strip():
        return repr(t)
    return t


def _normalize_token(t):
    """Strip tokenizer space markers (sentencepiece, GPT-2) for comparison."""
    return t.replace("\u2581", "").replace("\u0120", "").strip()


def load_logit_lens(layer, pos, prefix=""):
    """Load logit lens .pt file. Returns (top_k_probs, top_k_indices, inv_probs, inv_indices)."""
    return torch.load(
        LAYER_DIRS[layer] / f"{prefix}logit_lens_pos_{pos}.pt", weights_only=True
    )


def decode_tokens(indices):
    return [tokenizer.decode([int(i)]) for i in indices]


def load_patchscope(layer, pos, prefix=""):
    """Load auto_patch_scope .pt file. Returns dict with tokens_at_best_scale, selected_tokens, etc."""
    return torch.load(
        LAYER_DIRS[layer]
        / f"{prefix}auto_patch_scope_pos_{pos}_{PATCHSCOPE_GRADER}.pt",
        weights_only=False,
    )


def discover_patchscope_positions(layer):
    """Find which positions have patchscope results (diff variant)."""
    positions = []
    for f in sorted(
        LAYER_DIRS[layer].glob(f"auto_patch_scope_pos_*_{PATCHSCOPE_GRADER}.pt")
    ):
        m = re.search(r"auto_patch_scope_pos_(\d+)_", f.name)
        if m:
            positions.append(int(m.group(1)))
    return positions


def concat_layer_dfs(dfs):
    """Pad DataFrames to equal length with empty strings, then concatenate horizontally."""
    max_len = max(len(df) for df in dfs)
    padded = []
    for df in dfs:
        if len(df) < max_len:
            pad = pd.DataFrame(
                {col: [""] * (max_len - len(df)) for col in df.columns},
                index=range(len(df), max_len),
            )
            df = pd.concat([df, pad], axis=0)
        padded.append(df)
    return pd.concat(padded, axis=1)


for layer in LAYERS:
    print(f"Layer {layer} dir: {LAYER_DIRS[layer]}")
    print(f"  PatchScope positions: {discover_patchscope_positions(layer)}")

Layer 7 dir: ../../../adl_results/workspace/model-organisms/diffing_results/olmo2_1B_sft/examples_narrow-sft-2/activation_difference_lens/layer_7/tulu-3-sft-olmo-2-mixture
  PatchScope positions: [0, 1, 2, 3, 4, 5]
Layer 14 dir: ../../../adl_results/workspace/model-organisms/diffing_results/olmo2_1B_sft/examples_narrow-sft-2/activation_difference_lens/layer_14/tulu-3-sft-olmo-2-mixture
  PatchScope positions: [0, 1, 2, 3, 4, 5]
Layer 15 dir: ../../../adl_results/workspace/model-organisms/diffing_results/olmo2_1B_sft/examples_narrow-sft-2/activation_difference_lens/layer_15/tulu-3-sft-olmo-2-mixture
  PatchScope positions: [0, 1, 2, 3, 4, 5]


## 1. Logit Lens Analysis

### 1A. Single Position

Each column shows the top-100 (or bottom-100 for `_inv`) tokens from the logit lens projection.  
Format: `token (softmax_prob)`

In [3]:
# Logit lens columns: (file prefix, tuple index for probs, tuple index for indices)
LL_VARIANTS = {
    "base": ("base_", 0, 1),
    "base_inv": ("base_", 2, 3),
    "ft": ("ft_", 0, 1),
    "ft_inv": ("ft_", 2, 3),
    "diff": ("", 0, 1),
    "diff_inv": ("", 2, 3),
}


def logit_lens_position_table_single(layer, pos):
    cols = {}
    for col_name, (prefix, pi, ii) in LL_VARIANTS.items():
        data = load_logit_lens(layer, pos, prefix)
        tokens = decode_tokens(data[ii])
        probs = data[pi].tolist()
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(p)})" for t, p in zip(tokens, probs)
        ]
    df = pd.DataFrame(cols)
    if LOGIT_LENS_MAX_ROWS is not None:
        df = df.head(LOGIT_LENS_MAX_ROWS)
    return df


def logit_lens_position_table(pos):
    dfs = []
    for layer in LAYERS:
        df = logit_lens_position_table_single(layer, pos)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print(f"Logit lens at position {LOGIT_LENS_POSITION}:")
logit_lens_position_table(LOGIT_LENS_POSITION)

Logit lens at position -1:


layer_7                                                    \
                      base             base_inv                           ft   
0          .Today (0.0439)      urrenc (0.0288)               oller (0.0216)   
1       Buccane (7.17e-03)       pos (8.79e-03)           Buccane (6.81e-03)   
2       .Second (5.95e-03)       act (8.24e-03)          concrete (4.82e-03)   
3         /Area (5.58e-03)    askell (4.70e-03)              .Abs (4.00e-03)   
4         /Math (4.49e-03)      gons (3.13e-03)              fern (3.75e-03)   
5           aru (4.21e-03)     fácil (2.52e-03)           current (3.31e-03)   
6     /problems (3.85e-03)        دي (1.95e-03)         generally (2.75e-03)   
7      /respond (3.60e-03)    Phones (1.84e-03)              iger (2.21e-03)   
8          fter (3.19e-03)      anth (1.79e-03)            .There (2.14e-03)   
9           .au (3.08e-03)      ejec (1.67e-03)         according (2.08e-03)   
10     /problem (2.99e-03)  essional (1.57e-03)        engagement (1.95e-03)   
11    /entities (2.72e-03)      azon (1.57e-03)           studies (1.95e-03)   
12    /operator (2.64e-03)     emies (1.43e-03)       sovereignty (1.95e-03)   
13   confidence (2.56e-03)       dbl (1.43e-03)               cly (1.95e-03)   
14    belonging (2.56e-03)       med (1.43e-03)             isión (1.77e-03)   
15         oire (2.40e-03)     posix (1.39e-03)         depending (1.56e-03)   
16       soever (2.40e-03)         د (1.34e-03)             would (1.56e-03)   
17         ilot (2.12e-03)      Vers (1.30e-03)               riv (1.52e-03)   
18    /activity (2.00e-03)        �� (1.23e-03)   notwithstanding (1.47e-03)   
19   /community (1.87e-03)      orst (1.23e-03)              ream (1.47e-03)   

                                                                         \
                ft_inv                    diff                 diff_inv   
0        spos (0.0112)            mon (0.0118)           ource (0.0742)   
1   ategori (7.69e-03)          domin (0.0111)           neh (9.46e-03)   
2    urrenc (6.77e-03)          ' \n' (0.0107)   cellspacing (6.50e-03)   
3       pos (6.38e-03)            � (6.74e-03)          iore (5.74e-03)   
4       pon (4.12e-03)   influences (5.77e-03)       ---\n\n (5.37e-03)   
5     aland (2.66e-03)          ach (5.58e-03)           ICA (3.48e-03)   
6     suger (2.49e-03)            次 (4.91e-03)         /tags (3.27e-03)   
7     carga (2.49e-03)        oller (4.21e-03)           ioc (3.27e-03)   
8   ponsors (2.20e-03)            ( (3.97e-03)         rosse (3.07e-03)   
9    istema (2.06e-03)       master (3.83e-03)           eto (3.07e-03)   
10     acja (1.94e-03)      impacts (3.17e-03)         verts (3.07e-03)   
11    fácil (1.94e-03)      pressed (3.17e-03)         AMILY (2.88e-03)   
12     culo (1.82e-03)            @ (2.90e-03)        /lists (2.88e-03)   
13    abril (1.82e-03)         Gold (2.81e-03)       bgcolor (2.55e-03)   
14     pons (1.72e-03)          rid (2.67e-03)         marzo (2.24e-03)   
15     THEN (1.72e-03)            . (2.55e-03)           asu (2.24e-03)   
16   askell (1.72e-03)          riv (2.47e-03)       normals (2.24e-03)   
17    mostr (1.72e-03)      current (2.44e-03)     _requires (2.11e-03)   
18      agi (1.72e-03)          100 (2.15e-03)          ouch (2.11e-03)   
19      era (1.61e-03)            d (2.15e-03)          EDGE (1.98e-03)   

                layer_14                                           \
                    base               base_inv                ft   
0            To (0.9180)          zoek (0.7500)        1 (0.3242)   
1           The (0.0427)      contador (0.2148)      The (0.3047)   
2            In (0.0131)             메 (0.0227)        I (0.1436)   
3           1 (9.58e-03)         иск (5.74e-03)       To (0.0601)   
4         Let (3.74e-03)     Produto (4.46e-03)       As (0.0530)   
5           A (3.10e-03)     Detalle (1.42e-05)       In (0.0342)   
6           I (2.41e-03)           � (1.26e-05)     It 

In [4]:
# Logit lens columns: (file prefix, tuple index for probs, tuple index for indices)
LL_VARIANTS = {
    "base": ("base_", 0, 1),
    "base_inv": ("base_", 2, 3),
    "ft": ("ft_", 0, 1),
    "ft_inv": ("ft_", 2, 3),
    "diff": ("", 0, 1),
    "diff_inv": ("", 2, 3),
}


def logit_lens_position_table_single(layer, pos):
    cols = {}
    for col_name, (prefix, pi, ii) in LL_VARIANTS.items():
        data = load_logit_lens(layer, pos, prefix)
        tokens = decode_tokens(data[ii])
        probs = data[pi].tolist()
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(p)})" for t, p in zip(tokens, probs)
        ]
    df = pd.DataFrame(cols)
    if LOGIT_LENS_MAX_ROWS is not None:
        df = df.head(LOGIT_LENS_MAX_ROWS)
    return df


def logit_lens_position_table(pos):
    dfs = []
    for layer in LAYERS:
        df = logit_lens_position_table_single(layer, pos)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print(f"Logit lens at position {5}:")
logit_lens_position_table(5)

Logit lens at position 5:


layer_7                                                \
                       base             base_inv                       ft   
0         /problem (0.0342)       .vn (7.54e-03)        /problem (0.0198)   
1        /entities (0.0227)      sagt (5.19e-03)       /entities (0.0128)   
2        /problems (0.0166)        że (4.85e-03)       /problems (0.0116)   
3         .Today (9.77e-03)       ]]; (4.30e-03)          '\n' (5.31e-03)   
4        /global (9.77e-03)       (us (4.03e-03)       /manage (4.85e-03)   
5        /layout (8.12e-03)       ')" (3.14e-03)        .Round (4.27e-03)   
6        /manage (7.87e-03)      -ves (2.94e-03)       /global (4.27e-03)   
7           /job (7.14e-03)    testim (2.94e-03)         scarc (3.66e-03)   
8      /provider (6.32e-03)     zeigt (2.78e-03)       /layout (3.43e-03)   
9   /preferences (5.22e-03)       (!! (2.61e-03)      /logging (3.33e-03)   
10   /connection (4.76e-03)       ($. (2.44e-03)        .Today (3.23e-03)   
11    WHATSOEVER (4.49e-03)     lesen (2.30e-03)         :\n\n (3.23e-03)   
12        .Round (4.33e-03)     feliz (2.03e-03)  /environment (3.04e-03)   
13      /logging (3.94e-03)   kontrol (1.91e-03)      /account (3.04e-03)   
14       /crypto (3.71e-03)     spons (1.91e-03)     /provider (2.59e-03)   
15       /dialog (3.59e-03)    spiele (1.68e-03)   /categories (2.55e-03)   
16      /weather (3.17e-03)     scrut (1.68e-03)   /connection (2.37e-03)   
17          /reg (3.17e-03)       )": (1.49e-03)        /basic (2.32e-03)   
18      libertin (3.17e-03)       fas (1.49e-03)        /legal (2.29e-03)   
19       /entity (3.17e-03)      -git (1.40e-03)          /job (2.15e-03)   

                                                                            \
                   ft_inv                   diff                  diff_inv   
0            .vn (0.0113)             — (0.1973)             icap (0.0114)   
1           że (7.29e-03)         ' \n' (0.0222)            >[] (6.10e-03)   
2          ($. (6.44e-03)    Contract (5.10e-03)       perience (5.07e-03)   
3        spons (4.70e-03)       Dimit (4.64e-03)               (4.76e-03)   
4          (!! (4.70e-03)        obao (4.24e-03)             aż (4.76e-03)   
5         -ves (4.15e-03)         PPP (2.90e-03)         anmeld (4.49e-03)   
6        scrut (4.15e-03)          Pe (2.82e-03)  "\r\n\r\n\r\n (4.21e-03)   
7         helf (3.91e-03)        ████ (2.73e-03)            neh (3.07e-03)   
8       testim (3.04e-03)        whom (2.73e-03)            iez (2.88e-03)   
9     ,,,,,,,, (3.04e-03)       opper (2.73e-03)    >\n\n\n\n\n (2.88e-03)   
10   possibile (3.04e-03)          .. (2.41e-03)             уж (2.88e-03)   
11        sagt (3.04e-03)     ' \n\n' (2.41e-03)            fic (2.55e-03)   
12     personn (2.85e-03)         bai (2.26e-03)    ...\n\n\n\n (2.40e-03)   
13         -ie (2.85e-03)        fran (2.26e-03)            aes (2.40e-03)   
14       asign (2.69e-03)       bonds (2.12e-03)         [image (2.24e-03)   
15         bmi (2.37e-03)    ———————— (1.94e-03)  advertisement (2.11e-03)   
16       zeigt (2.37e-03)         kel (1.94e-03)     -translate (2.11e-03)   
17    protagon (2.37e-03)   contracts (1.82e-03)           icie (2.11e-03)   
18       lesen (2.09e-03)        abet (1.82e-03)          verse (1.98e-03)   
19         )": (1.97e-03)          ,— (1.76e-03)          ***\n (1.98e-03)   

            layer_14                                           \
                base              base_inv                 ft   
0         , (0.6445)     contador (0.6133)         , (0.4863)   
1       and (0.1211)           �� (0.0144)       ' ' (0.1631)   
2       the (0.0947)           �� (0.0144)       the (0.1118)   
3        in (0.0601)        subur (0.0112)       and (0.1084)   
4       ' ' (0.0454)           bö (0.0112)        in (0.0771)   
5         a (0.0123)    kontrol (6.81e-03)         a (0.0164)   
6      to (3.59e-03)       samt (6.81e-03)      to (4.91e-03)   
7      of (3.28e-03)   

### 1B. Aggregated Across All Positions

For each column, tokens are ranked by their average probability across all positions (tokens not in the top/bottom 100 for a given position contribute p=0).  
Format: `token (avg_prob)`

In [5]:
def logit_lens_aggregated_single(layer):
    agg = {}
    for col_name, (prefix, pi, ii) in LL_VARIANTS.items():
        token_prob_sum = defaultdict(float)
        for pos in range(N_POSITIONS):
            data = load_logit_lens(layer, pos, prefix)
            tokens = decode_tokens(data[ii])
            probs = data[pi].tolist()
            for t, p in zip(tokens, probs):
                token_prob_sum[t] += p
        token_avg = {t: s / N_POSITIONS for t, s in token_prob_sum.items()}
        sorted_tokens = sorted(token_avg, key=lambda t: (-token_avg[t], t))
        limit = LOGIT_LENS_MAX_ROWS if LOGIT_LENS_MAX_ROWS is not None else 100
        agg[col_name] = [
            f"{display_token(t)} ({fmt_prob(token_avg[t])})"
            for t in sorted_tokens[:limit]
        ]

    max_len = max(len(v) for v in agg.values())
    for k in agg:
        agg[k] += [""] * (max_len - len(agg[k]))
    return pd.DataFrame(agg)


def logit_lens_aggregated():
    dfs = []
    for layer in LAYERS:
        df = logit_lens_aggregated_single(layer)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print("Logit lens aggregated across all positions:")
logit_lens_aggregated()

Logit lens aggregated across all positions:


layer_7                                                 \
                       base              base_inv                       ft   
0        /entities (0.0244)          .vn (0.0184)        /problem (0.0119)   
1         /problem (0.0126)    /Register (0.0113)       /entities (0.0118)   
2      /problems (9.38e-03)       sagt (7.61e-03)       /problems (0.0117)   
3        /global (7.29e-03)     testim (6.95e-03)             , (5.73e-03)   
4      /provider (6.62e-03)        -ie (6.12e-03)     /provider (5.29e-03)   
5    /connection (6.34e-03)      asign (5.02e-03)   /connection (4.72e-03)   
6         .Today (5.81e-03)         że (4.75e-03)      /account (4.29e-03)   
7        /manage (5.35e-03)      zeigt (4.47e-03)       /global (3.66e-03)   
8   /environment (4.79e-03)    personn (3.53e-03)       /manage (3.51e-03)   
9      /customer (3.97e-03)       -ves (3.30e-03)  /environment (3.43e-03)   
10  /preferences (3.59e-03)      probs (3.18e-03)  /preferences (3.18e-03)   
11       /layout (3.58e-03)       roku (2.84e-03)          .Abs (2.97e-03)   
12       /shared (3.40e-03)     ):\n\n (2.55e-03)       perpetr (2.67e-03)   
13       /dialog (3.37e-03)          ť (2.52e-03)             — (2.64e-03)   
14      /account (3.21e-03)       elig (2.49e-03)             / (2.50e-03)   
15      libertin (3.11e-03)        )": (2.42e-03)       /dialog (2.38e-03)   
16             , (3.03e-03)        esl (2.33e-03)     /customer (2.37e-03)   
17         .Take (3.01e-03)      lesen (2.22e-03)        /basic (2.27e-03)   
18       /entity (2.98e-03)   ,,,,,,,, (2.16e-03)       /entity (2.19e-03)   
19        /legal (2.83e-03)   misunder (2.11e-03)   /categories (2.19e-03)   

                                                                              \
                  ft_inv                      diff                  diff_inv   
0           .vn (0.0164)                — (0.2642)                � (0.0308)   
1     /Register (0.0137)                ■ (0.0202)                 (0.0241)   
2           -ie (0.0102)                ■ (0.0109)           )`\n (3.54e-03)   
3    misunder (8.85e-03)    ' \n\n\n\n\n' (0.0103)          début (3.29e-03)   
4      testim (6.21e-03)         ———————— (0.0100)            oad (2.60e-03)   
5       probs (5.71e-03)        />.\n\n (8.70e-03)    ...\n\n\n\n (2.58e-03)   
6        sagt (5.21e-03)            ..\ (7.19e-03)             `_ (2.42e-03)   
7       scrut (5.10e-03)             —— (5.62e-03)        oplevel (2.03e-03)   
8        elig (4.94e-03)          ' \n' (4.56e-03)            >[] (1.85e-03)   
9       asign (4.68e-03)  <|endoftext|> (4.15e-03)           icap (1.81e-03)   
10         że (4.47e-03)        ' \n\n' (3.43e-03)              � (1.70e-03)   
11    personn (3.72e-03)             ,— (3.36e-03)            ﻿\n (1.59e-03)   
12       helf (3.70e-03)           ———— (3.30e-03)  advertisement (1.55e-03)   
13   ,,,,,,,, (3.64e-03)              ț (2.71e-03)           >{\n (1.44e-03)   
14       -ves (3.59e-03)           ████ (2.47e-03)          `\r\n (1.43e-03)   
15      zeigt (3.32e-03)              ¬ (2.43e-03)          poste (1.34e-03)   
16   protagon (2.95e-03)           agos (2.39e-03)            );\ (1.33e-03)   
17      lesen (2.61e-03)             .. (2.32e-03)              ъ (1.25e-03)   
18          ť (2.50e-03)           pond (2.17e-03)          ...\n (1.21e-03)   
19     ):\n\n (2.29e-03)              ‚ (2.16e-03)              ‑ (1.18e-03)   

             layer_14                                           \
                 base              base_inv                 ft   
0          , (0.8575)     contador (0.9290)         , (0.6939)   
1        ' ' (0.0778)     testim (3.10e-03)       ' ' (0.2344)   
2        the (0.0283)      subur (2.42e-03)       and (0.0256)   
3        and (0.0235)   misunder (2.41e-03)       the (0.0251)   
4       in (5.08e-03)    kontrol (2.19e-03)      in (6.73e-03)   
5        ( (2.76e-03)       helf (1.89e-03)       ( (4.81e-03)   
6        a (1.

## 2. PatchScope Analysis

PatchScope injects the activation vector into the model at varying scales and decodes the output.  
Unlike logit lens, there are no inverse variants -- only `base`, `ft`, and `diff`.  
Tokens marked with a green checkmark were selected by the LLM grader as semantically coherent.

### 2A. Single Position

Shows tokens at the best scale found by the auto patch scope search.  
Format: `token (prob)` with `\u2705` if in `selected_tokens`

In [6]:
PS_VARIANTS = [("base", "base_"), ("ft", "ft_"), ("diff", "")]


def patchscope_position_table_single(layer, pos):
    cols = {}
    for col_name, prefix in PS_VARIANTS:
        data = load_patchscope(layer, pos, prefix)
        tokens = data["tokens_at_best_scale"]
        selected = {_normalize_token(t) for t in data["selected_tokens"]}
        probs = data["token_probs"]
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(p)})"
            + (" \u2705" if _normalize_token(t) in selected else "")
            for t, p in zip(tokens, probs)
        ]

    max_len = max(len(v) for v in cols.values())
    for k in cols:
        cols[k] += [""] * (max_len - len(cols[k]))
    return pd.DataFrame(cols)


def patchscope_position_table(pos):
    dfs = []
    for layer in LAYERS:
        df = patchscope_position_table_single(layer, pos)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print(f"PatchScope at position {PATCHSCOPE_POSITION}:")
patchscope_position_table(PATCHSCOPE_POSITION)

PatchScope at position -1:


layer_7                                                   \
                       base                         ft                  diff   
0           .Today (0.0433)             oller (0.0221)          ' ' (0.0964)   
1            aru (6.36e-03)      concrete (5.40e-03) ✅          ... (0.0266)   
2        /Area (5.17e-03) ✅         Buccane (5.14e-03)      thank (0.0238) ✅   
3        Buccane (4.71e-03)     generally (4.38e-03) ✅         anna (0.0169)   
4        .Second (4.56e-03)       current (3.67e-03) ✅      Thank (0.0101) ✅   
5        /Math (3.98e-03) ✅            fern (3.64e-03)   thanks (8.58e-03) ✅   
6           fter (3.78e-03)            .Abs (3.42e-03)     target (8.53e-03)   
7     confidence (3.66e-03)       studies (2.77e-03) ✅    Thank (8.37e-03) ✅   
8            .au (3.51e-03)     according (2.69e-03) ✅         th (8.34e-03)   
9           ilot (2.85e-03)         would (2.66e-03) ✅         42 (7.84e-03)   
10   /problems (2.77e-03) ✅            iger (2.59e-03)          I (7.61e-03)   
11     belonging (2.52e-03)          .There (2.34e-03)         na (6.55e-03)   
12    /respond (2.27e-03) ✅          wouldn (2.22e-03)         ow (5.78e-03)   
13    /problem (2.22e-03) ✅       depends (2.00e-03) ✅    thank (5.39e-03) ✅   
14   /entities (1.94e-03) ✅     depending (1.98e-03) ✅          < (4.87e-03)   
15            ,a (1.90e-03)             cly (1.87e-03)          p (4.20e-03)   
16   /operator (1.84e-03) ✅    engagement (1.73e-03) ✅         if (4.07e-03)   
17          oire (1.75e-03)             riv (1.65e-03)   Thanks (3.93e-03) ✅   
18   /activity (1.73e-03) ✅   sovereignty (1.64e-03) ✅       that (3.72e-03)   
19  /community (1.68e-03) ✅      contrary (1.63e-03) ✅          n (3.57e-03)   

                layer_14                                                      \
                    base                          ft                    diff   
0            To (0.9414)                The (0.1849)        Thank (0.1060) ✅   
1           The (0.0284)                 As (0.1699)            >\n (0.0603)   
2       Let (9.83e-03) ✅                 To (0.1562)        Based (0.0196) ✅   
3          In (6.08e-03)                  I (0.1436)          thank (0.0153)   
4           1 (5.25e-03)             Sure (0.1165) ✅         Dear (0.0143) ✅   
5         ### (2.33e-03)                  1 (0.1029)        Sorry (0.0126) ✅   
6          ** (1.02e-03)             Here (0.0203) ✅        Great (0.0119) ✅   
7           I (9.12e-04)        Certainly (0.0172) ✅            > (7.20e-03)   
8        Sure (8.76e-04)            Thank (0.0109) ✅            I (6.77e-03)   
9   Certainly (8.05e-04)            Based (9.58e-03)           >> (6.23e-03)   
10          A (6.26e-04)               In (7.46e-03)         ok (4.65e-03) ✅   
11         As (5.76e-04)               It (7.16e-03)     Notify (4.36e-03) ✅   
12         It (3.81e-04)            There (5.13e-03)     excuse (3.86e-03) ✅   
13       We (3.57e-04) ✅          Great (4.72e-03) ✅  Regarding (3.86e-03) ✅   
14       This (2.96e-04)  Unfortunately (3.99e-03) ✅      Thank (3.40e-03) ✅   
15        For (2.78e-04)            Yes (3.51e-03) ✅            ( (3.19e-03)   
16    First (2.78e-04) ✅             This (3.10e-03)  According (3.19e-03) ✅   
17    Given (1.52e-04) ✅            Let (2.75e-03) ✅        thank (3.19e-03)   
18     Here (1.00e-04) ✅              For (2.33e-03)        Yes (3.19e-03) ✅   
19    There (9.63e-05) ✅          Sorry (1.30e-03) ✅     Please (3.01e-03) ✅   

                layer_15                                                  
                    base                        ft                  diff  
0            To (0.8555)                I (0.2168)          >\n (0.4355)  
1           The (0.0483)              The (0.1914)      Thank (0.0359) ✅  
2         Let (0.0201) ✅               As (0.1914)        thank (0.0170)  
3            ** (0.0178)               To (0.1162)      Great (0.0170) ✅  
4             1 (0.0157)           Sure (0.1025)

### 2B. Aggregated Across All PatchScope Positions

Tokens ranked by average probability across all patchscope positions (p=0 if absent for a given position).  
Green checkmark if the token was in `selected_tokens` for **any** position.  
Format: `token (avg_prob)`

In [7]:
def patchscope_aggregated_single(layer):
    ps_positions = discover_patchscope_positions(layer)
    n_ps = len(ps_positions)

    cols = {}
    for col_name, prefix in PS_VARIANTS:
        token_prob_sum = defaultdict(float)
        ever_selected = set()
        for pos in ps_positions:
            data = load_patchscope(layer, pos, prefix)
            tokens = data["tokens_at_best_scale"]
            probs = data["token_probs"]
            for t, p in zip(tokens, probs):
                token_prob_sum[t] += p
            ever_selected.update(_normalize_token(t) for t in data["selected_tokens"])

        token_avg = {t: s / n_ps for t, s in token_prob_sum.items()}
        sorted_tokens = sorted(token_avg, key=lambda t: (-token_avg[t], t))
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(token_avg[t])})"
            + (" \u2705" if _normalize_token(t) in ever_selected else "")
            for t in sorted_tokens
        ]

    max_len = max(len(v) for v in cols.values())
    for k in cols:
        cols[k] += [""] * (max_len - len(cols[k]))
    return pd.DataFrame(cols)


def patchscope_aggregated():
    dfs = []
    for layer in LAYERS:
        df = patchscope_aggregated_single(layer)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


ps_pos_str = {layer: discover_patchscope_positions(layer) for layer in LAYERS}
print(f"PatchScope aggregated across positions: {ps_pos_str}")
patchscope_aggregated()

PatchScope aggregated across positions: {7: [0, 1, 2, 3, 4, 5], 14: [0, 1, 2, 3, 4, 5], 15: [0, 1, 2, 3, 4, 5]}


layer_7                             \
                         base                         ft   
0         /problem (0.0278) ✅        /problem (0.0220) ✅   
1        /problems (0.0196) ✅       /problems (0.0153) ✅   
2        /entities (0.0191) ✅                's (0.0122)   
3          /manage (0.0156) ✅       /entities (0.0110) ✅   
4         .Today (7.70e-03) ✅       /manage (9.31e-03) ✅   
5      /provider (4.83e-03) ✅               , (5.53e-03)   
6        /global (4.60e-03) ✅         solve (5.27e-03) ✅   
7   /preferences (3.81e-03) ✅             the (5.09e-03)   
8        /layout (2.95e-03) ✅               I (4.97e-03)   
9    /connection (2.57e-03) ✅    understand (4.79e-03) ✅   
10      /logging (2.30e-03) ✅         start (3.97e-03) ✅   
11          /job (2.26e-03) ✅            '\n' (3.44e-03)   
12        /tasks (2.10e-03) ✅             you (3.01e-03)   
13       /crypto (1.87e-03) ✅     summarize (2.86e-03) ✅   
14       /object (1.83e-03) ✅       /global (2.67e-03) ✅   
15        /legal (1.60e-03) ✅       clarify (2.66e-03) ✅   
16       /shared (1.49e-03) ✅      /logging (2.64e-03) ✅   
17      /effects (1.43e-03) ✅            John (2.54e-03)   
18     /activity (1.40e-03) ✅  /environment (2.20e-03) ✅   
19           /pr (1.29e-03) ✅      /account (2.16e-03) ✅   
20      /respond (1.19e-03) ✅       /shared (2.02e-03) ✅   
21      /company (1.05e-03) ✅         begin (1.86e-03) ✅   
22          .Round (1.05e-03)              we (1.83e-03)   
23  /controllers (9.96e-04) ✅     /provider (1.69e-03) ✅   
24           /pl (9.38e-04) ✅   /connection (1.56e-03) ✅   
25      /vendors (7.57e-04) ✅            bear (1.53e-03)   
26      /testing (6.80e-04) ✅           scarc (1.49e-03)   
27         /disc (6.71e-04) ✅  /preferences (1.48e-03) ✅   
28            /con (5.94e-04)          .Round (1.44e-03)   
29      WHATSOEVER (5.71e-04)          /inc (1.41e-03) ✅   
30              ет (5.69e-04)          .Today (1.41e-03)   
31             iez (5.39e-04)       analyze (1.32e-03) ✅   
32       /engine (5.35e-04) ✅              to (1.25e-03)   
33  /environment (5.35e-04) ✅       address (1.20e-03) ✅   
34          /reg (5.35e-04) ✅        answer (1.18e-03) ✅   
35  /application (5.24e-04) ✅       proceed (1.14e-03) ✅   
36      /general (5.15e-04) ✅          find (1.02e-03) ✅   
37       /dialog (5.11e-04) ✅              in (9.71e-04)   
38     /customer (5.03e-04) ✅        /basic (9.27e-04) ✅   
39         /spec (4.93e-04) ✅       /layout (9.27e-04) ✅   
40       /entity (4.92e-04) ✅        /legal (8.34e-04) ✅   
41            /man (4.92e-04)           :\n\n (7.36e-04)   
42        libertin (4.24e-04)               : (4.66e-04)   
43             aac (3.71e-04)        concerns (4.54e-04)   
44         present (3.63e-04)          '\n\n' (4.50e-04)   
45        formally (3.17e-04)               . (4.04e-04)   
46             ise (3.17e-04)         /apis (4.02e-04) ✅   
47      externally (3.08e-04)    /functions (3.92e-04) ✅   
48                                          [ (3.84e-04)   
49                                      /disc (3.69e-04)   
50                                 /respond (3.69e-04) ✅   
51                                  /events (3.62e-04) ✅   
52                                    /conf (3.54e-04) ✅   
53                                 /context (3.49e-04) ✅   
54                                /activity (3.36e-04) ✅   
55                                          / (3.29e-04)   
56                                        /re (3.26e-04)   
57                                  /create (3.04e-04) ✅   
58                                                         
59                                                         
60                                                         
61                                                         
62                                                         
63                                                         
64                                                         
65

## 3. Diff Logit Lens Across Positions

Shows only the **diff** variant of the logit lens for selected positions across all layers.
Format: `token (softmax_prob)`

In [8]:
DIFF_POSITIONS = [-3, -1, 0, 1, 2, 3, 10, 50, 100]


def logit_lens_diff_positions_table():
    """Show diff logit lens across multiple positions for all layers."""
    dfs = []
    for layer in LAYERS:
        col_data = {}
        for pos in DIFF_POSITIONS:
            prefix, pi, ii = LL_VARIANTS["diff"]
            data = load_logit_lens(layer, pos, prefix)
            tokens = decode_tokens(data[ii])
            probs = data[pi].tolist()
            col = [f"{display_token(t)} ({fmt_prob(p)})" for t, p in zip(tokens, probs)]
            if LOGIT_LENS_MAX_ROWS is not None:
                col = col[:LOGIT_LENS_MAX_ROWS]
            col_data[f"pos_{pos}"] = col
        layer_df = pd.DataFrame(col_data)
        layer_df.columns = pd.MultiIndex.from_product(
            [[f"layer_{layer}"], layer_df.columns]
        )
        dfs.append(layer_df)
    return concat_layer_dfs(dfs)


print(f"Logit lens DIFF across positions: {DIFF_POSITIONS}")
logit_lens_diff_positions_table()

Logit lens DIFF across positions: [-3, -1, 0, 1, 2, 3, 10, 50, 100]


layer_7                                                  \
                     pos_-3                  pos_-1                   pos_0   
0           bone (9.95e-03)            mon (0.0118)             .. (0.0128)   
1          chein (7.26e-03)          domin (0.0111)          mon (8.00e-03)   
2            ikh (6.23e-03)          ' \n' (0.0107)        Olymp (7.29e-03)   
3           Sche (5.68e-03)            � (6.74e-03)          sur (6.65e-03)   
4            iid (5.00e-03)   influences (5.77e-03)    statement (5.86e-03)   
5            ную (3.66e-03)          ach (5.58e-03)     Archives (4.85e-03)   
6            Alg (3.54e-03)            次 (4.91e-03)         ster (4.85e-03)   
7     Hutchinson (3.23e-03)        oller (4.21e-03)   statements (3.66e-03)   
8    Legislation (2.85e-03)            ( (3.97e-03)         olon (3.66e-03)   
9          också (2.85e-03)       master (3.83e-03)            — (3.66e-03)   
10            __ (2.76e-03)      impacts (3.17e-03)        iston (3.45e-03)   
11        oplast (2.59e-03)      pressed (3.17e-03)     lections (3.34e-03)   
12         uture (2.59e-03)            @ (2.90e-03)       fabric (3.23e-03)   
13        shapes (2.52e-03)         Gold (2.81e-03)          Ana (3.04e-03)   
14      Telegram (2.52e-03)          rid (2.67e-03)        arton (2.94e-03)   
15        mycket (2.44e-03)            . (2.55e-03)          abh (2.59e-03)   
16        inject (2.37e-03)          riv (2.47e-03)          akh (2.52e-03)   
17           lio (2.29e-03)      current (2.44e-03)         iphy (2.44e-03)   
18         press (2.21e-03)          100 (2.15e-03)         Hipp (2.44e-03)   
19           alg (2.15e-03)            d (2.15e-03)          Sur (2.15e-03)   

                                                                               \
                    pos_1                     pos_2                     pos_3   
0              — (0.0654)                — (0.3320)                — (0.4746)   
1       Contract (0.0121)         Contract (0.0107)           auté (5.80e-03)   
2        opper (6.90e-03)          ' \n' (8.30e-03)          ' \n' (4.09e-03)   
3            _ (4.73e-03)           auté (3.57e-03)          —\n\n (2.33e-03)   
4     lections (4.30e-03)      contracts (3.36e-03)             .. (2.26e-03)   
5     caratter (3.80e-03)  ' \n\n\n\n\n' (2.87e-03)          Dimit (2.00e-03)   
6    contracts (3.80e-03)           whom (2.87e-03)              — (1.94e-03)   
7           .. (3.05e-03)          abled (2.46e-03)       Contract (1.94e-03)   
8        ' \n' (3.05e-03)           able (2.38e-03)              ■ (1.66e-03)   
9          mon (2.87e-03)             .. (2.23e-03)            ..\ (1.46e-03)   
10    Archives (2.78e-03)          —\n\n (1.97e-03)  ' \n\n\n\n\n' (1.34e-03)   
11         den (2.46e-03)          Dimit (1.97e-03)           whom (1.34e-03)   
12       Olymp (2.38e-03)           ████ (1.97e-03)          opper (1.29e-03)   
13       iston (2.03e-03)       contract (1.91e-03)        bindung (1.29e-03)   
14           < (2.03e-03)       Archives (1.91e-03)          pueda (1.29e-03)   
15   Aristotle (2.03e-03)              ■ (1.85e-03)      contracts (1.14e-03)   
16        urga (1.85e-03)             *& (1.85e-03)          bonds (1.11e-03)   
17         ..\ (1.85e-03)            PHA (1.74e-03)            bay (1.07e-03)   
18      uomini (1.74e-03)          marty (1.69e-03)       contract (1.01e-03)   
19      martyr (1.74e-03)          pueda (1.63e-03)             ,— (1.01e-03)   

                                                                              \
                      pos_10                  pos_50                 pos_100   
0                 — (0.3730)              — (0.3262)              — (0.2305)   
1             ' \n' (0.0120)              ■ (0.0162)              ■ (0.0293)   
2         ' \n\n' (7.05e-03)  <|endoftext|> (0.0143)              ■ (0.0156)   
3               ■ (5.49e-03)       ———————— (0.0112)  ' \n\n\n\n\n' (0.0147)   
4              .. 

## 4. Diff PatchScope Across Positions

Shows only the **diff** variant of PatchScope for selected positions across all layers.
Format: `token (prob)` with `✅` if in `selected_tokens`

In [9]:
PS_DIFF_POSITIONS = [-3, -1, 0, 1, 2, 3]


def patchscope_diff_positions_table():
    """Show diff patchscope across multiple positions for all layers."""
    dfs = []
    for layer in LAYERS:
        col_data = {}
        for pos in PS_DIFF_POSITIONS:
            try:
                data = load_patchscope(layer, pos, prefix="")
            except FileNotFoundError:
                col_data[f"pos_{pos}"] = ["(not available)"]
                continue
            tokens = data["tokens_at_best_scale"]
            selected = {_normalize_token(t) for t in data["selected_tokens"]}
            probs = data["token_probs"]
            col = [
                f"{display_token(t)} ({fmt_prob(p)})"
                + (" ✅" if _normalize_token(t) in selected else "")
                for t, p in zip(tokens, probs)
            ]
            col_data[f"pos_{pos}"] = col
        layer_df = pd.DataFrame({k: pd.Series(v) for k, v in col_data.items()}).fillna(
            ""
        )
        layer_df.columns = pd.MultiIndex.from_product(
            [[f"layer_{layer}"], layer_df.columns]
        )
        dfs.append(layer_df)
    return concat_layer_dfs(dfs)


print(f"PatchScope DIFF across positions: {PS_DIFF_POSITIONS}")
patchscope_diff_positions_table()

PatchScope DIFF across positions: [-3, -1, 0, 1, 2, 3]


layer_7                                              \
                       pos_-3                pos_-1                 pos_0   
0             bone (9.24e-03)          ' ' (0.0964)       blue (0.0249) ✅   
1            chein (6.41e-03)          ... (0.0266)           -> (0.0142)   
2             Sche (6.29e-03)      thank (0.0238) ✅          ann (0.0136)   
3              iid (5.72e-03)         anna (0.0169)          ' ' (0.0135)   
4              ikh (5.72e-03)      Thank (0.0101) ✅      red (9.79e-03) ✅   
5              ную (4.27e-03)   thanks (8.58e-03) ✅       name (7.01e-03)   
6            Alg (4.05e-03) ✅     target (8.53e-03)   yellow (6.80e-03) ✅   
7     Hutchinson (3.80e-03) ✅    Thank (8.37e-03) ✅      world (6.33e-03)   
8               __ (3.22e-03)         th (8.34e-03)        bye (6.04e-03)   
9            uture (3.13e-03)         42 (7.84e-03)       '\n' (5.08e-03)   
10         press (3.00e-03) ✅          I (7.61e-03)        day (5.05e-03)   
11      Telegram (2.78e-03) ✅         na (6.55e-03)        ana (4.98e-03)   
12       Posting (2.64e-03) ✅         ow (5.78e-03)    green (4.92e-03) ✅   
13           också (2.62e-03)    thank (5.39e-03) ✅        day (4.61e-03)   
14             alg (2.62e-03)          < (4.87e-03)         en (4.55e-03)   
15          shapes (2.46e-03)          p (4.20e-03)         un (4.53e-03)   
16            Tear (2.38e-03)         if (4.07e-03)    message (4.30e-03)   
17            alon (2.33e-03)   Thanks (3.93e-03) ✅      hello (4.24e-03)   
18   Legislation (2.21e-03) ✅       that (3.72e-03)      world (4.10e-03)   
19            enty (2.19e-03)          n (3.57e-03)    brown (3.84e-03) ✅   

                                                                              \
                   pos_1                     pos_2                     pos_3   
0          anna (0.0635)                — (0.0677)               -> (0.1148)   
1           ' ' (0.0457)            ' \n' (0.0107)           blue (0.0573) ✅   
2        blue (0.0347) ✅     Contract (8.95e-03) ✅             anna (0.0417)   
3       world (0.0205) ✅           whom (5.38e-03)              113 (0.0409)   
4           ann (0.0153)    contracts (5.32e-03) ✅          green (0.0370) ✅   
5        target (0.0125)           able (3.98e-03)              ' ' (0.0340)   
6        '\n' (9.56e-03)          abled (3.32e-03)              ana (0.0225)   
7          -> (7.94e-03)           auté (3.06e-03)             '\n' (0.0166)   
8      target (6.73e-03)          PPP (2.87e-03) ✅               -> (0.0117)   
9       sky (5.02e-03) ✅          Aud (2.79e-03) ✅          green (0.0106) ✅   
10    welcome (4.33e-03)  ' \n\n\n\n\n' (2.70e-03)               an (0.0104)   
11      sea (4.16e-03) ✅          PPP (2.65e-03) ✅            ann (9.34e-03)   
12       name (3.63e-03)             .. (2.54e-03)       yellow (6.84e-03) ✅   
13          h (3.60e-03)          PHA (2.34e-03) ✅             is (6.56e-03)   
14        man (3.59e-03)            bay (2.34e-03)          red (5.66e-03) ✅   
15          p (3.49e-03)     Archives (2.32e-03) ✅              a (5.39e-03)   
16   yellow (3.44e-03) ✅            aqu (2.30e-03)            ann (4.72e-03)   
17       Anna (3.40e-03)        bonds (2.29e-03) ✅  <|endoftext|> (4.71e-03)   
18    goodbye (3.35e-03)     contract (2.17e-03) ✅              [ (4.27e-03)   
19          l (3.30e-03)          pueda (2.02e-03)         blue (3.93e-03) ✅   

              layer_14                                                     \
                pos_-3                  pos_-1                      pos_0   
0           | (0.9531)        Thank (0.1060) ✅         formula (0.0207) ✅   
1           > (0.0175)            >\n (0.0603)             bia (9.38e-03)   
2        >> (8.24e-03)        Based (0.0196) ✅           adera (8.12e-03)   
3         » (2.98e-03)          thank (0.0153)   calculation (4.91e-03) ✅   
4         ＞ (2.04e-03)         Dear (0.0143) ✅            bedo (3.38e-03)   
5        >| (1.19e-03)     